## How to reproduce this notebook (supplementary)

3D registration / alignment of consecutive liver sections. Companion to
`run_Lynx_3D.ipynb`; see [data/README.md](../../../data/README.md) for inputs under
`data/xenium_3d/`.

In [ ]:
# --- LYNX reproducibility: repo root + central paths + LYNX import ------------
import os as _os, sys as _sys
try:
    _d = _os.path.dirname(_os.path.abspath(__file__))
except NameError:  # interactive / jupytext: __file__ undefined
    _d = _os.getcwd()
while _d != _os.path.dirname(_d) and not _os.path.exists(_os.path.join(_d, "paths.py")):
    _d = _os.path.dirname(_d)
if _d not in _sys.path:
    _sys.path.insert(0, _d)
from paths import DATA_DIR, RESULTS_DIR, FIGURES_DIR, FINAL_FIGURES_DIR
import lynx  # noqa: F401 - registers models/ & util/ on sys.path
# -----------------------------------------------------------------------------

## 3D tissue alignment of consecutive sections

Interactive attempts with napari-spatialdata

- (1). Xenium iterative alignment against the middle fixed image
- (2). DESI-Xenium alignment (layer-to-layer)

In [1]:
%gui qt

In [2]:
import os
import sys
import cv2
import gc
import gzip
import json
import tifffile

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt

from skimage.filters import gaussian as gaussian_blur
from skimage.exposure import equalize_adapthist
from skimage.transform import rescale

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from IPython.display import display

sns.set_context('paper')
rcParams.update({'font.family': 'Arial'})
rcParams.update({'font.size': 12})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

from util import IO, registration

%matplotlib inline
%load_ext autoreload
%autoreload 2


/home/yinuo/Desktop/azizi_lab/IMS/env/lib/python3.9/site-packages/numba/core/decorators.py:282: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [ ]:
import spatialdata as sd
import spatialdata_io as sdio
import spatialdata_plot as sdpl
from napari_spatialdata import Interactive

from spatialdata import SpatialData
from spatialdata.models import ShapesModel, TableModel, Image2DModel
from spatialdata.transformations import (
    BaseTransformation,
    Sequence,
    get_transformation,
    set_transformation,
    remove_transformation
)

sc.set_figure_params(scanpy=True, fontsize=10)
rcParams.update({'figure.dpi': 200})
rcParams.update({'axes.grid': False})


### Xenium cross-section alignment

#### Load data

In [13]:
XENIUM_PATH = DATA_DIR + '/xenium_3d'
SAMPLE_ID = 'NIH_F5'
IMAGE_KEY = 'morphology_focus'

# Define fixed and moving section IDs
fixed_section = 'section_04'
moving_section = 'section_03'



In [ ]:
# Load xenium adatas and save as zarr
# fixed_sdata = sdio.xenium(
#     path=os.path.join(XENIUM_PATH, SAMPLE_ID, fixed_section),
#     morphology_mip=False,
#     morphology_focus=True
# )
# moving_sdata = sdio.xenium(
#     path=os.path.join(XENIUM_PATH, SAMPLE_ID, moving_section),
#     morphology_mip=False,
#     morphology_focus=True
# )
# IO.load_spatial_metadata(fixed_sdata['table'])

moving_sdata = sdio.xenium(
    path=os.path.join(XENIUM_PATH, SAMPLE_ID, moving_section),
    morphology_mip=False,
    morphology_focus=True
)
IO.load_spatial_metadata(moving_sdata['table'])

# fixed_outdir = os.path.join(XENIUM_PATH, SAMPLE_ID, fixed_section, 'sdata.zarr')
# fixed_sdata.write(fixed_outdir, overwrite=True)
moving_outdir = os.path.join(XENIUM_PATH, SAMPLE_ID, moving_section, 'sdata.zarr')
moving_sdata.write(moving_outdir, overwrite=True)

del moving_sdata
gc.collect()

In [ ]:
# Load xenium data
if os.path.exists(os.path.join(XENIUM_PATH, SAMPLE_ID, fixed_section, 'sdata.zarr')) and \
   os.path.exists(os.path.join(XENIUM_PATH, SAMPLE_ID, moving_section, 'sdata.zarr')):
    print("Zarr files already exist. Loading directly.")
    fixed_sdata = sd.read_zarr(os.path.join(XENIUM_PATH, SAMPLE_ID, fixed_section, 'sdata1.zarr'))
    moving_sdata = sd.read_zarr(os.path.join(XENIUM_PATH, SAMPLE_ID, moving_section, 'sdata1.zarr'))

else:
    fixed_sdata = sdio.xenium(
        path=os.path.join(XENIUM_PATH, SAMPLE_ID, fixed_section),
        morphology_mip=False,
        morphology_focus=True
    )
    moving_sdata = sdio.xenium(
        path=os.path.join(XENIUM_PATH, SAMPLE_ID, moving_section),
        morphology_mip=False,
        morphology_focus=True
    )
    IO.load_spatial_metadata(fixed_sdata['table'])
    IO.load_spatial_metadata(moving_sdata['table'])

# Extract fixed image size limits (execute only once!!!)
scalefactor = 0.2125
fixed_height, fixed_width = 5072, 4247
# fixed_height, fixed_width = sd.get_pyramid_levels(fixed_sdata[IMAGE_KEY], n=0).to_numpy().shape[-2:]
# fixed_height, fixed_width = int(fixed_height*scalefactor), int(fixed_width*scalefactor)
# fixed_height, fixed_width

# Filter "aligned" coordinate system from moving_sdata if exists
# registration.delete_coordinate_system(moving_sdata, "aligned")

Zarr files already exist. Loading directly.


[2026-03-24 23:48:33] root_attr: multiscales
[2026-03-24 23:48:33] root_attr: omero
[2026-03-24 23:48:33] root_attr: spatialdata_attrs
[2026-03-24 23:48:33] datasets [{'coordinateTransformations': [{'scale': [1.0, 1.0, 1.0], 'type': 'scale'}], 'path': '0'}, {'coordinateTransformations': [{'scale': [1.0, 2.000083794201441, 2.0001000700490343], 'type': 'scale'}], 'path': '1'}, {'coordinateTransformations': [{'scale': [1.0, 4.000167588402882, 4.000600480384308], 'type': 'scale'}], 'path': '2'}, {'coordinateTransformations': [{'scale': [1.0, 8.00167616493463, 8.001200960768616], 'type': 'scale'}], 'path': '3'}, {'coordinateTransformations': [{'scale': [1.0, 16.008718980549965, 16.00240192153723], 'type': 'scale'}], 'path': '4'}]
[2026-03-24 23:48:33] resolution: 0
[2026-03-24 23:48:33]  - shape ('c', 'y', 'x') = (5, 23869, 19987)
[2026-03-24 23:48:33]  - chunks =  ['1', '4096 (+ 3389)', '4096 (+ 3603)']
[2026-03-24 23:48:33]  - dtype = uint16
[2026-03-24 23:48:33] resolution: 1
[2026-03-24

**Male fixed (reference dim):**

**Female fixed (reference) dim: `(5072, 4247)`**

In [ ]:
# Visualize gene expression before alignment
sq.pl.spatial_scatter(
    fixed_sdata['table'],
    color='CYP3A4',
    size=20,
    img=False
)

sq.pl.spatial_scatter(
    moving_sdata['table'],
    color='CYP3A4',
    size=20,
    img=False
)


#### (Optional): data subsetting

---
E.g. Subset sdata into independent slices

In [ ]:
moving_sdata['morphology_focus']['scale0']

In [ ]:
# Optional preprocess if necessary:
# (e.g. divide into smaller subsections)

_, ydim, xdim = moving_sdata['morphology_focus']['scale0'].image.shape
x1_min, x1_max = 0, xdim // 2
x2_min, x2_max = xdim // 2, xdim
ymin, ymax = 0, ydim

moving_sdata1 = sd.bounding_box_query(
    moving_sdata, 
    axes=('y', 'x'),
    min_coordinate=[ymin, x1_min],
    max_coordinate=[ymax, x1_max],
    target_coordinate_system="global"
)

moving_sdata2 = sd.bounding_box_query(
    moving_sdata, 
    axes=('y', 'x'),
    min_coordinate=[ymin, x2_min],
    max_coordinate=[ymax, x2_max],
    target_coordinate_system="global"
)

moving_sdata1.pl.render_images(
    elements="morphology_focus",
    scale="scale4",
    channel=0
).pl.show()

moving_sdata2.pl.render_images(
    elements="morphology_focus",
    scale="scale4",
    channel=0
).pl.show()

gc.collect()

In [ ]:
moving_outdir = os.path.join(XENIUM_PATH, SAMPLE_ID, moving_section, 'sdata1.zarr')
moving_sdata1.write(moving_outdir, overwrite=True)
subsection_outdir = os.path.join(XENIUM_PATH, SAMPLE_ID, 'section_05', 'sdata.zarr')
moving_sdata2.write(subsection_outdir, overwrite=True)

---

#### Alignment

In [17]:
Interactive(fixed_sdata)
Interactive(moving_sdata)

2026-03-24 23:49:08.343 | INFO     | napari_spatialdata._view:_on_layer_update:355 - Updating layer.
2026-03-24 23:49:08.344 | INFO     | napari_spatialdata._view:_on_layer_update:355 - Updating layer.
2026-03-24 23:49:23.116 | INFO     | napari_spatialdata._view:_on_layer_update:355 - Updating layer.
2026-03-24 23:49:23.119 | INFO     | napari_spatialdata._view:_on_layer_update:355 - Updating layer.
2026-03-24 23:49:23.119 | INFO     | napari_spatialdata._view:_on_layer_update:355 - Updating layer.


Align moving --> fixed image by affine transformation w.r.t landmarks:
- Create new point layers & rename them respectively (e.g. `fixed` & `moving`)
- Add fixed & moving landmarks
- Linking annotation layers to each `sdata` (`Shift + L`)
- Saving annotations to each `sdata` (`Shift + E`)

In [ ]:
if 'landmarks' in fixed_sdata:
    del fixed_sdata['landmarks']
if 'landmarks' in moving_sdata:
    del moving_sdata['landmarks']
    
# Apply transformation to fixed landmarks themselves (global -> aligned)
if 'aligned' in fixed_sdata.coordinate_systems:
    fixed_landmarks = ShapesModel.parse(
        registration.transform_coords(
            fixed_sdata,
            fixed_sdata['fixed_landmarks'].compute().values,
            image_key=IMAGE_KEY
        ),
        geometry=0,
        radius=10, 
    )
else:
    fixed_landmarks = ShapesModel.parse(
        fixed_sdata['fixed_landmarks'].compute().values,
        geometry=0,
        radius=10, 
    )
fixed_sdata['landmarks'] = fixed_landmarks

moving_landmarks = ShapesModel.parse(
    moving_sdata['moving_landmarks'].compute().values,
    geometry=0,
    radius=10
)
moving_sdata['landmarks'] = moving_landmarks


Affine transformation w/ landmarks on the chosen image layer:

In [ ]:
affine = registration.align_elements_using_landmarks(
    references_coords=fixed_sdata['landmarks'],
    reference_element=fixed_sdata[IMAGE_KEY],
    reference_coordinate_system='aligned' if 'aligned' in fixed_sdata.coordinate_systems else 'global',
    moving_coords=moving_sdata['landmarks'],
    moving_element=moving_sdata[IMAGE_KEY],
    moving_coordinate_system='global',
    new_coordinate_system='aligned'
)

affine

Aligning the rest of `moving_sdata` elements:

In [ ]:
registration.postpone_transformation(
    sdata=moving_sdata,
    transformation=affine.transformations[-1],
    source_coordinate_system="global",
    target_coordinate_system="aligned",
)


In [ ]:
# Double check alignment in napari
Interactive(fixed_sdata)
Interactive(moving_sdata)

Update associated adata.obsm['spatial'] from the moving sdata

In [ ]:
# Optional: load moving_sdata coordinates from cells.csv.gz
# coords_df = pd.read_csv(os.path.join(XENIUM_PATH, SAMPLE_ID, moving_section, 'cells.csv.gz'), compression='gzip')
# coords_df = coords_df[['cell_id', 'x_centroid', 'y_centroid']]
# coords_df.set_index('cell_id', inplace=True)
# coords_df = coords_df.loc[moving_sdata.table.obs['cell_id']]

# moving_sdata.table.obsm['spatial'] = coords_df.values


registration.update_adata_coords(
    moving_sdata, 
    matrix=affine.transformations[-1].to_affine_matrix(input_axes=('x', 'y'), output_axes=('x', 'y')),
    size_limit=(fixed_width, fixed_height)
)


In [ ]:
sc.set_figure_params(scanpy=True, fontsize=10)
sq.pl.spatial_scatter(
    fixed_sdata['table'],
    color='CYP3A4',
    size=20,
    img=False
)

sq.pl.spatial_scatter(
    moving_sdata['table'],
    color='CYP3A4',
    size=20,
    img=False
)


In [ ]:
# Remove landmarks after alignment
del fixed_sdata['fixed_landmarks'], moving_sdata['moving_landmarks']

# Save aligned objects to sdata zarr objects
moving_outdir = os.path.join(XENIUM_PATH, SAMPLE_ID, moving_section, 'sdata1.zarr')
moving_sdata.write(moving_outdir, overwrite=True)

Overwrite original sdata `zarr` object w/ aligned features
```Python
%%bash -s {XENIUM_PATH}/{SAMPLE_ID}/{MOVING_SECTION}
mv $1/sdata1.zarr $1/sdata.zarr
```


#### Post-alignment steps

- Filter common (x, y) frames

In [ ]:
XENIUM_PATH = DATA_DIR + '/xenium_3d'
SAMPLE_ID = 'NIH_F5'
IMAGE_KEY = 'morphology_focus'

sections = sorted([
    f for f in os.listdir(os.path.join(XENIUM_PATH, SAMPLE_ID))
    if os.path.isdir(os.path.join(XENIUM_PATH, SAMPLE_ID, f))
])
sections

In [ ]:
sdatas = []
for section in sections:
    sdata = sd.read_zarr(os.path.join(XENIUM_PATH, SAMPLE_ID, section, 'sdata.zarr'))
    sdatas.append(sdata)

In [ ]:
# Filter common `sdata.table.obsm['spatial']` coordinates across all sections
xmin = max([sdata.table.obsm['spatial'][:, 0].min() for sdata in sdatas])
xmax = min([sdata.table.obsm['spatial'][:, 0].max() for sdata in sdatas])
ymin = max([sdata.table.obsm['spatial'][:, 1].min() for sdata in sdatas])
ymax = min([sdata.table.obsm['spatial'][:, 1].max() for sdata in sdatas])

xmin, xmax, ymin, ymax

In [ ]:
adatas = []
for i in range(len(sdatas)):
    adata = sdatas[i].table[
            ((sdatas[i].table.obsm['spatial'][:, 0] >= xmin) &
            (sdatas[i].table.obsm['spatial'][:, 0] < xmax) &
            (sdatas[i].table.obsm['spatial'][:, 1] >= ymin) &
            (sdatas[i].table.obsm['spatial'][:, 1] < ymax))
    ].copy()
    adatas.append(adata)
    del sdatas[i].table

    sdatas[i].table = adata
    sdatas[i].write(os.path.join(XENIUM_PATH, SAMPLE_ID, sections[i], 'sdata1.zarr'), overwrite=True)

del adata
gc.collect()

### DESI alignment

- (TODO): still need to hard-align across positive & negative channels
- Positive-negative combination per-slide
- Generate `adata / sdata` objects

In [ ]:
# Helper functions
from spatialdata.models import ShapesModel

import xml.etree.ElementTree as ET
import scipy.ndimage as ndi
from skimage.filters import threshold_otsu
from skimage.filters import gaussian as gaussian_blur
from skimage.morphology import binary_erosion, disk

def norm_by_channel(x):
    x_normed = np.zeros_like(x, dtype=np.float32)
    for i, chan in enumerate(x):
        x_normed[i] = (chan-chan.min())/(chan.max()-chan.min())
    return x_normed


def remove_holes(roi, min_area):
    r""" Remove holes & FP lslands in binary ROI mask"""
    roi_filtered = roi.copy().astype(np.uint8)
    roi_labeled, n_features = ndi.label(roi)
    
    for i in range(1, n_features+1):
        if (roi_labeled == i).sum() < min_area:
            roi_filtered[roi_labeled == i] = 0
            
    return ndi.binary_fill_holes(roi_filtered).astype(np.uint8)

def get_roi_mask(
    img: np.ndarray, 
    sigma: float = 5.,
    erode_pixel=5,
    min_area: float = 0.
):
    r"""
    Compute binary matrix for ROI by filtering out 
    background & boundary artifact pixels
    """

    def __apply_otsu_threshold(array):
        thresh = threshold_otsu(array)
        return array > thresh

    # `img` dim: [Y, X] or [C, Y, X]
    img_blurred = img.copy() if img.ndim == 2 \
                  else img.mean(0)  

    if sigma == 0. and erode_pixel == 0:
        return np.ones_like(img_blurred, dtype=bool)

    img_blurred = gaussian_blur(img_blurred, sigma=sigma)
    roi_mask = __apply_otsu_threshold(img_blurred)
    if min_area > 0:
        roi_mask = remove_holes(roi_mask, min_area)

    roi_mask = binary_erosion(
        image=roi_mask,
        footprint=disk(radius=erode_pixel)
    )
    return roi_mask


def load_desi_tif(
    filename, 
    sigma=5, 
    erode_pixel=5, 
    min_area=500,
):
    r"""
    Load raw DESI tif image as AnnData object
    """
    filename = filename if 'tif' in filename else filename + '.ome.tif'
    assert os.path.exists(filename), \
         "DESI file {} doesn't exist".format(filename)

    img = norm_by_channel(tifffile.imread(filename))  # dim: [C, Y, X]

    # Load raw image, filter out background & tissue border outliers
    roi_mask = get_roi_mask(img, sigma=sigma, erode_pixel=erode_pixel, min_area=min_area)
    adata = sc.AnnData(img[:, roi_mask].T)
    IO.load_spatial_metadata(adata, load_img=False)
    adata.uns['X_img'] = np.einsum('cyx, yx -> cyx', img, roi_mask)
    
    coords = np.asarray(np.nonzero(roi_mask))  # YX-index, dim: [2, Y*X]
    adata.obs['x_centroid'], adata.obs['y_centroid'] = coords[1], coords[0]
    adata.obsm['spatial'] = np.array([coords[1], coords[0]]).T  # XY-index

    # Load feature annotations
    try:
        mz_labels = IO.load_ome_labels(filename)
        adata.var_names = mz_labels
    except ET.ParseError:
        pass

    return adata, img


# Optional rotations / flipping
def flip_image(img_coord_tuple, axis='y'):
    """Flip image & coordinates along the cartesian axis (xy-based)"""
    img_raw, coords_raw = img_coord_tuple
    img_flipped = img_raw.copy()
    coords_flipped = coords_raw.copy()
    
    if axis == 'x':
        # Flip along x-axis (vertical flip)
        if img_raw.ndim == 3:  # [C, Y, X]
            img_flipped = np.flip(img_raw, axis=1)
            y_max = img_raw.shape[1] - 1
        else:  # [Y, X]
            img_flipped = np.flip(img_raw, axis=0)
            y_max = img_raw.shape[0] - 1
        
        # Transform y coordinates: new_y = y_max - old_y
        coords_flipped[:, 1] = y_max - coords_raw[:, 1]
    else:
        # Flip along y-axis (horizontal flip)
        if img_raw.ndim == 3:  # [C, Y, X]
            img_flipped = np.flip(img_raw, axis=2)
            x_max = img_raw.shape[2] - 1
        else:  # [Y, X]
            img_flipped = np.flip(img_raw, axis=1)
            x_max = img_raw.shape[1] - 1
        
        # Transform x coordinates: new_x = x_max - old_x
        coords_flipped[:, 0] = x_max - coords_raw[:, 0]
    
    return img_flipped, coords_flipped


def transpose_image(img_coord_tuple):
    img_raw, coords_raw = img_coord_tuple
    if img_raw.ndim == 3:  # [C, Y, X] -> [C, X, Y]
        img_transposed = np.transpose(img_raw, (0, 2, 1))
    else:  # [Y, X] -> [X, Y]
        img_transposed = np.transpose(img_raw, (1, 0))
    
    # Swap x and y coordinates: (x, y) -> (y, x)
    coords_transposed = coords_raw[:, [1, 0]]
    
    return img_transposed, coords_transposed


def concat_adatas(adata1, adata2, batch_key='ion', batch_categories=['+', '-']):
    """Concatenate two AnnData objects based on common coordinates in `obsm['spatial']`"""
    coords1 = np.array([tuple(coord) for coord in adata1.obsm['spatial']])
    coords2 = np.array([tuple(coord) for coord in adata2.obsm['spatial']])
    
    common_coords = set(map(tuple, coords1)).intersection(set(map(tuple, coords2)))
    common_coords = np.array(list(common_coords)).astype(int)

    idx1 = [np.where((coords1 == coord).all(axis=1))[0][0] for coord in common_coords]
    idx2 = [np.where((coords2 == coord).all(axis=1))[0][0] for coord in common_coords]
    
    adata1_sub = adata1[idx1].copy()
    adata2_sub = adata2[idx2].copy()

    del adata1_sub.obs, adata1_sub.uns
    del adata2_sub.obs, adata2_sub.uns

    adata1_sub.obsm['spatial'] = adata1_sub.obsm['spatial'].astype(int)
    adata2_sub.obsm['spatial'] = adata2_sub.obsm['spatial'].astype(int)
    
    
    # Concatenate along var axis, add `var` batch info 
    adata_intersect = ad.AnnData(
        X=np.concatenate([adata1_sub.X, adata2_sub.X], axis=1),
        var=pd.concat([adata1_sub.var, adata2_sub.var]),
        obsm={'spatial': adata1_sub.obsm['spatial']}
    )
    adata_intersect.var[batch_key] = batch_categories[0]*len(adata1_sub.var) + \
                                     batch_categories[1]*len(adata2_sub.var)
    
    # Add dummy spatial object to `uns`
    IO.load_spatial_metadata(adata_intersect)

    return adata_intersect

In [ ]:
DESI_PATH = DATA_DIR + '/desi_3d'
SAMPLE_ID = 'NIH_F5'
DESI_POS_SUFFIX = '_pos_40um'
DESI_NEG_SUFFIX = '_neg_40um'
IMAGE_KEY = 'desi'

OUTDIR = os.path.join(DESI_PATH, SAMPLE_ID+'_'+DESI_POS_SUFFIX.split('_')[-1])
if not os.path.exists(OUTDIR):
    os.makedirs(OUTDIR, exist_ok=True)

# fixed_section = 'section_01'


#### Preprocess data

In [ ]:
adata_desi_pos, img_pos = load_desi_tif(
    filename=os.path.join(DESI_PATH, SAMPLE_ID + DESI_POS_SUFFIX, fixed_section),
    erode_pixel=0
)

adata_desi_neg, img_neg = load_desi_tif(
    filename=os.path.join(DESI_PATH, SAMPLE_ID + DESI_NEG_SUFFIX, fixed_section),
    erode_pixel=0
)

fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.imshow(img_pos.mean(0), cmap='magma')
ax1.axis('off')
ax2.imshow(img_neg.mean(0), cmap='magma')
ax2.axis('off')
plt.show()

sq.pl.spatial_scatter(
    adata_desi_pos, 
    color=adata_desi_pos.var_names[20],
    size=1,
    img=False,
    cmap='magma'
)

sq.pl.spatial_scatter(
    adata_desi_neg, 
    color=adata_desi_neg.var_names[20],
    size=1,
    img=False,
    cmap='magma'
)

In [ ]:
# Check whether spatial transformation is as desired
img_pos_transformed, coords_pos_transformed = flip_image(
    flip_image(
        transpose_image(
            (img_pos, adata_desi_pos.obsm['spatial']), 
        ),
        axis='x'
    ),
    axis='y'
)

img_neg_transformed, coords_neg_transformed = flip_image(
    (img_neg, adata_desi_neg.obsm['spatial']),
    axis='y'
)


fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.imshow(img_pos_transformed.mean(0), cmap='magma')
ax1.axis('off')
ax2.imshow(img_neg_transformed.mean(0), cmap='magma')
ax2.axis('off')
plt.show()

In [ ]:
adata_desi_pos.obsm['spatial'] = coords_pos_transformed
adata_desi_pos.uns['X_img'] = img_pos_transformed

adata_desi_neg.obsm['spatial'] = coords_neg_transformed
adata_desi_neg.uns['X_img'] = img_neg_transformed

sq.pl.spatial_scatter(
    adata_desi_pos, 
    color=adata_desi_pos.var_names[20],
    size=1,
    img=False,
    cmap='magma'
)

sq.pl.spatial_scatter(
    adata_desi_neg, 
    color=adata_desi_neg.var_names[20],
    size=1,
    img=False,
    cmap='magma'
)

Save raw image & adata as SpatialData objects:

In [ ]:
# Positive mode
pos_sdata = registration.compile_spatialdata(
    adata=adata_desi_pos,
    img=img_pos_transformed,
    image_key=IMAGE_KEY
)

pos_sdata.write(
    os.path.join(DESI_PATH, SAMPLE_ID + DESI_POS_SUFFIX, fixed_section+'_sdata.zarr'),
    overwrite=True
)


# Negative mode
neg_sdata = registration.compile_spatialdata(
    adata=adata_desi_neg,
    img=img_neg_transformed,
    image_key=IMAGE_KEY
)

neg_sdata.write(
    os.path.join(DESI_PATH, SAMPLE_ID + DESI_NEG_SUFFIX, fixed_section+'_sdata.zarr'),
    overwrite=True
)

gc.collect()

Validation across sections:

In [ ]:
filenames = sorted([
    f for f in os.listdir(os.path.join(DESI_PATH, SAMPLE_ID + DESI_POS_SUFFIX))
    if os.path.isdir(os.path.join(DESI_PATH, SAMPLE_ID + DESI_POS_SUFFIX, f))
])


for filename in filenames:
    pos_sdata = sd.read_zarr(
        os.path.join(DESI_PATH, SAMPLE_ID + DESI_POS_SUFFIX, filename)
    )
    neg_sdata = sd.read_zarr(
        os.path.join(DESI_PATH, SAMPLE_ID + DESI_NEG_SUFFIX, filename)
    )
    
    sq.pl.spatial_scatter(
        pos_sdata.table, 
        color=pos_sdata.table.var_names[20],
        size=1,
        img=False,
        cmap='magma'
    )
    sq.pl.spatial_scatter(
        neg_sdata.table, 
        color=neg_sdata.table.var_names[20],
        size=1,
        img=False,
        cmap='magma'
    )


gc.collect()



#### Alignment

- `fixed_sdata`: `+` ion; `moving_sdata`: `-` ion
- Attach both `-` images & transformed coords to `+` sdata

In [ ]:
fixed_section = 'section_09'

fixed_sdata = sd.read_zarr(
    os.path.join(DESI_PATH, SAMPLE_ID + DESI_POS_SUFFIX, fixed_section+'_sdata1.zarr')
)

moving_sdata = sd.read_zarr(
    os.path.join(DESI_PATH, SAMPLE_ID + DESI_NEG_SUFFIX, fixed_section+'_sdata1.zarr')
)

fixed_height, fixed_width = sd.get_pyramid_levels(fixed_sdata[IMAGE_KEY], n=0).to_numpy().shape[-2:]
scalefactor = 1.0

In [ ]:
Interactive(fixed_sdata)
Interactive(moving_sdata)

In [ ]:
if 'landmarks' in fixed_sdata:
    del fixed_sdata['landmarks']
if 'landmarks' in moving_sdata:
    del moving_sdata['landmarks']
    
fixed_landmarks = ShapesModel.parse(
    fixed_sdata['fixed_landmarks'].compute().values,
    geometry=0,
    radius=1, 
)
fixed_sdata['landmarks'] = fixed_landmarks

moving_landmarks = ShapesModel.parse(
    moving_sdata['moving_landmarks'].compute().values,
    geometry=0,
    radius=1
)
moving_sdata['landmarks'] = moving_landmarks


In [ ]:
affine = align_elements_using_landmarks(
    references_coords=fixed_sdata['landmarks'],
    reference_element=fixed_sdata[IMAGE_KEY],
    reference_coordinate_system='global',
    moving_coords=moving_sdata['landmarks'],
    moving_element=moving_sdata[IMAGE_KEY],
    moving_coordinate_system='global',
    new_coordinate_system='aligned'
)

affine

In [ ]:
# Validate alignment in napari
gc.collect()
Interactive(fixed_sdata)
Interactive(moving_sdata)

In [ ]:
registration.update_adata_coords(
    moving_sdata, 
    matrix=affine.transformations[-1].to_affine_matrix(input_axes=('x', 'y'), output_axes=('x', 'y')),
    size_limit=(fixed_width, fixed_height),
    scalefactor=scalefactor
)

Append aligned image & coordinates across `+/-` modes, save new SpatialData object:

In [ ]:
M = affine.transformations[-1].to_affine_matrix(input_axes=('x', 'y'), output_axes=('x', 'y'))[:-1]
fixed_img = fixed_sdata.images[IMAGE_KEY]['scale0']['image'].to_numpy()
moving_img = moving_sdata.images[IMAGE_KEY]['scale0']['image'].to_numpy()
transformed_img = []
for chan in range(moving_img.shape[0]):
    transformed_chan = registration.affine_warp(
        moving_img[chan],
        dst_shape=(fixed_height, fixed_width),
        M=M
    )
    transformed_img.append(transformed_chan)
transformed_img = np.array(transformed_img)


concat_img = np.concatenate([fixed_img, transformed_img], axis=0)
concat_adata = concat_adatas(
    fixed_sdata.table,
    moving_sdata.table
)

concat_sdata = registration.compile_spatialdata(
    adata=concat_adata,
    img=concat_img,
    image_key=IMAGE_KEY
)


concat_sdata.write(
    os.path.join(OUTDIR, fixed_section+'_sdata.zarr'),
    overwrite=True
)

In [ ]:
Interactive(concat_sdata)

### Cross-modal alignment

- Fixed modality: `Xenium` (spatial transcriptomics)
- Moving modality: `DESI` (spatial metabolomics)
- Save the mapped coordinates under `sdata.table.obsm['*_map']` w/o actually warping images

In [ ]:
SAMPLE_ID = 'NIH_F5'
XENIUM_PATH = DATA_DIR + '/xenium_3d'
XENIUM_IMAGE_KEY = 'morphology_focus'

DESI_PATH = DATA_DIR + f'/desi_3d/{SAMPLE_ID}_40um'
DESI_IMAGE_KEY = 'desi'

FIXED_MAP_KEY = 'xenium_map'  # adata.obsm[key] to save xenium-to-desi mapping
MOVING_MAP_KEY = 'desi_map' # adata.obsm[key] to save desi-to-xenium mapping

scalefactor = 0.2125


In [ ]:
%ls {XENIUM_PATH}/{SAMPLE_ID}

In [ ]:
%ls {DESI_PATH}

In [ ]:
section = 'section_08'

fixed_sdata = sd.read_zarr(
    os.path.join(XENIUM_PATH, SAMPLE_ID, section, 'sdata.zarr')
)

moving_sdata = sd.read_zarr(
    os.path.join(DESI_PATH, section+'_sdata.zarr')
)

In [ ]:
Interactive(fixed_sdata)
Interactive(moving_sdata)

In [ ]:
if 'landmarks' in fixed_sdata:
    del fixed_sdata['landmarks']
if 'landmarks' in moving_sdata:
    del moving_sdata['landmarks']
    
fixed_landmarks = ShapesModel.parse(
    fixed_sdata['fixed_landmarks'].compute().values,
    geometry=0,
    radius=1, 
)    
fixed_sdata['landmarks'] = fixed_landmarks

moving_landmarks = ShapesModel.parse(
    moving_sdata['moving_landmarks'].compute().values,
    geometry=0,
    radius=1
)
moving_sdata['landmarks'] = moving_landmarks


In [ ]:
affine = align_elements_using_landmarks(
    references_coords=fixed_sdata['landmarks'],
    reference_element=fixed_sdata[XENIUM_IMAGE_KEY],
    reference_coordinate_system='aligned' if 'aligned' in fixed_sdata.coordinate_systems else 'global',
    moving_coords=moving_sdata['landmarks'],
    moving_element=moving_sdata[DESI_IMAGE_KEY],
    moving_coordinate_system='global',
    new_coordinate_system='multimodal_aligned'
)

affine

In [ ]:
# Validate alignment in napari
gc.collect()
Interactive(fixed_sdata)
Interactive(moving_sdata)

Update cross-modal registered coordinates under `sdata.table.obsm`

In [ ]:
def filter_mapped_coords(
    mapped_coords,
    original_coords,
    verbose=False
):
    """Filter mapped Xenium -> DESI coords"""
    coords_set = set(map(tuple, original_coords))
    coord_to_keep = np.array([
        coord if tuple(coord) in coords_set else [-1, -1]
        for coord in mapped_coords
    ])

    if verbose:
        n_mapped = np.all(coord_to_keep != [-1, -1], axis=1).sum()
        print(f"Number of mapped coordinates kept: {n_mapped}/{len(mapped_coords)}")

    return coord_to_keep

In [ ]:
M = affine.transformations[-1].to_affine_matrix(input_axes=('x', 'y'), output_axes=('x', 'y'))[:-1]

mapped_moving_coords = registration.affine_transform_coords(
    M=M, coords=moving_sdata.table.obsm['spatial']
) * scalefactor

mapped_fixed_coords = np.round(registration.inverse_affine_transform_coords(
    M=M, coords=fixed_sdata.table.obsm['spatial'] / scalefactor
)).astype(int)

# Filter Xenium coords that are mappable to
# DESI counterparts w/ valid integer coords.
mapped_fixed_coords = filter_mapped_coords(
    mapped_fixed_coords, 
    moving_sdata.table.obsm['spatial'], 
    verbose=True
)

fixed_sdata.table.obsm[MOVING_MAP_KEY] = mapped_fixed_coords
moving_sdata.table.obsm[FIXED_MAP_KEY] = mapped_moving_coords


In [ ]:
# Save spatialdata objects w/ updated mapping
fixed_sdata.write(
    os.path.join(XENIUM_PATH, SAMPLE_ID, section, 'sdata1.zarr'),
    overwrite=True
)

moving_sdata.write(
    os.path.join(DESI_PATH, section+'_sdata1.zarr'),
    overwrite=True
)

### Registration error

- Cross-modal alignment: residual at its own landmarks, saved w/ each section of the data release (`fixed_landmarks` in Xenium & `moving_landmarks` in DESI)
- Xenium cross-section alignment: its landmarks are removed after alignment, the residual is measured from the expression of consecutive sections

In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
import spatialdata as sd
from spatialdata.transformations import get_transformation
from IPython.display import display

RELEASE_PATH = DATA_DIR + '/LYNX_liver_dataset/NIH_F5_3D'
SAMPLE_ID = 'NIH_F5'
SECTIONS = [
   'section_01',
   'section_02',
   'section_04',
   'section_05',
   'section_06',
   'section_07',
   'section_08',
]
SPACING = 110.     # Depth between consecutive sections (um); section_03 is absent
PIXEL_SIZE = 0.2125  # Xenium image pixel (um)

outdir = RESULTS_DIR + '/liver_3d/'
if not os.path.exists(outdir):
    os.makedirs(outdir, exist_ok=True)
se = lambda values: np.std(values, ddof=1) / np.sqrt(len(values))

In [ ]:
# Cross-modal alignment: distance between a DESI landmark mapped by the saved transformation & its Xenium landmark
records = []
for section in SECTIONS:
    fixed_sdata = sd.read_zarr(
        os.path.join(RELEASE_PATH, 'xenium', section, 'sdata.zarr'), selection=('points',)
    )
    moving_sdata = sd.read_zarr(
        os.path.join(RELEASE_PATH, 'DESI', section+'_sdata.zarr'), selection=('points', 'images')
    )
    fixed_coords = fixed_sdata['fixed_landmarks'].compute()[['x', 'y']].to_numpy(float)   # aligned frame (image pixels)
    moving_coords = moving_sdata['moving_landmarks'].compute()[['x', 'y']].to_numpy(float)  # DESI pixels
    M = get_transformation(
        moving_sdata['desi'], to_coordinate_system='multimodal_aligned'
    ).to_affine_matrix(input_axes=('x', 'y'), output_axes=('x', 'y'))

    residuals = np.linalg.norm(
        (np.c_[moving_coords, np.ones(len(moving_coords))] @ M.T)[:, :2] - fixed_coords, axis=1
    ) * PIXEL_SIZE
    records += [{'section': section, 'landmark': i+1, 'residual_um': residual} for i, residual in enumerate(residuals)]

landmark_df = pd.DataFrame(records)
landmark_df.to_csv(os.path.join(outdir, f'LYNX_liver_{SAMPLE_ID}_registration_error.csv'), index=False)
section_df = landmark_df.groupby('section')['residual_um'].agg(['size', 'mean'])
display(section_df.round(1))
print(f"Mean over sections: {section_df['mean'].mean():.1f} ± {se(section_df['mean']):.1f} um")

In [ ]:
# Xenium cross-section alignment, measured from the tissue. For two consecutive sections in aligned coordinates:
# genes w/ >= 0.3 counts per cell in the fixed section (section_04) are binned (40 um for the coarse search, 20 um for the fine one),
# smoothed (sigma 20 um) & z-scored over the tissue. The second section is scaled & rotated about the tissue centre on a grid
# and shifted by the peak of the mean cross-correlation of the gene maps; the (scale, rotation, shift) w/ the highest correlation
# is the alignment preferred by the expression, and the distance it moves the cells from their aligned positions is the residual
import cv2
import scipy.sparse as sp
import scipy.fft as sfft

SIGMA, MAX_SHIFT, EXTENT = 20., 300., (4600., 5300.)

def load_cells(section):
    adata = ad.read_zarr(os.path.join(RELEASE_PATH, 'xenium', section, 'sdata.zarr', 'tables', 'table'))
    X = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
    return np.asarray(adata.obsm['spatial'], dtype=float), X.astype(np.float32)


def get_maps(coords, X, total, size):
    r"""z-scored smoothed gene maps (zero-padded for the cross-correlation) & the tissue mask"""
    W, H = int(EXTENT[0] / size), int(EXTENT[1] / size)
    FW, FH = (sfft.next_fast_len(n + int(MAX_SHIFT / size) + 1) for n in (W, H))
    ix, iy = (np.floor(coords[:, k] / size).astype(int) for k in (0, 1))
    valid = (ix >= 0) & (ix < W) & (iy >= 0) & (iy < H)
    bins = sp.csr_matrix(
        (np.ones(valid.sum(), np.float32), (iy[valid] * W + ix[valid], np.flatnonzero(valid))), shape=(H * W, len(coords))
    )
    counts = np.asarray(bins @ X).reshape(H, W, -1)
    density = cv2.GaussianBlur(np.asarray(bins @ total[:, None]).reshape(H, W), (0, 0), SIGMA / size)
    mask = density > 0.2 * np.median(density[density > 0])

    z, m = np.zeros((X.shape[1], FH, FW), np.float32), np.zeros((FH, FW), np.float32)
    for g in range(X.shape[1]):
        expr = np.log1p(
            1e4 * cv2.GaussianBlur(np.ascontiguousarray(counts[:, :, g]), (0, 0), SIGMA / size) / np.maximum(density, 1e-6)
        )
        z[g, :H, :W] = np.where(mask, (expr - expr[mask].mean()) / (expr[mask].std() + 1e-6), 0.)
    m[:H, :W] = mask
    return z, m


def get_shift(F_fixed, F_mask, z, m, size):
    r"""Shift at the peak of the mean cross-correlation (sub-bin by parabolic interpolation) -> (correlation, dx, dy)"""
    FH, FW = z.shape[1:]
    num = sfft.irfft2(F_fixed * np.conj(sfft.rfft2(z, workers=8)), s=(FH, FW), workers=8).mean(0)
    overlap = sfft.irfft2(F_mask * np.conj(sfft.rfft2(m)), s=(FH, FW))
    idx = np.r_[-int(MAX_SHIFT / size):int(MAX_SHIFT / size) + 1]
    corr = np.where(
        overlap[np.ix_(idx, idx)] > 0.5 * min(F_mask[0, 0].real, m.sum()), (num / np.maximum(overlap, 1.))[np.ix_(idx, idx)], -1.
    )
    j, i = np.unravel_index(np.argmax(corr), corr.shape)
    vertex = lambda u, v, w: 0. if abs(u - 2 * v + w) < 1e-12 else 0.5 * (u - w) / (u - 2 * v + w)
    dy, dx = float(idx[j]), float(idx[i])
    if 0 < j < len(idx) - 1 and 0 < i < len(idx) - 1:
        dy, dx = dy + vertex(corr[j - 1, i], corr[j, i], corr[j + 1, i]), dx + vertex(corr[j, i - 1], corr[j, i], corr[j, i + 1])
    return float(corr[j, i]), dx * size, dy * size


def move_coords(coords, center, scale, degree, shift=(0., 0.)):
    theta = np.radians(degree)
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    return center + scale * (coords - center) @ R.T + np.asarray(shift)


def get_residual(fixed_coords, fixed_X, moving_coords, moving_X):
    center, fixed_total, moving_total = fixed_coords.mean(0), fixed_X.sum(1), moving_X.sum(1)

    def get_scorer(size):
        z, m = get_maps(fixed_coords, fixed_X, fixed_total, size)
        F_fixed, F_mask = sfft.rfft2(z, workers=8), sfft.rfft2(m)
        return lambda scale, degree: get_shift(
            F_fixed, F_mask, *get_maps(move_coords(moving_coords, center, scale, degree), moving_X, moving_total, size), size
        )
    coarse, fine = get_scorer(40.), get_scorer(20.)
    grid = {
        (scale, degree): coarse(scale, degree)[0]
        for scale in np.round(np.arange(0.88, 1.121, 0.02), 3) for degree in np.arange(-8, 8.1, 2.)
    }
    scale0, degree0 = max(grid, key=grid.get)
    grid = {
        (scale, degree): fine(scale, degree)
        for scale in np.round(np.arange(scale0 - 0.015, scale0 + 0.0151, 0.005), 4)
        for degree in np.arange(degree0 - 1.5, degree0 + 1.51, 0.5)
    }
    (scale, degree), (_, dx, dy) = max(grid.items(), key=lambda item: item[1][0])
    displacement = np.linalg.norm(move_coords(moving_coords, center, scale, degree, (dx, dy)) - moving_coords, axis=1)
    return {
        'scale': float(scale), 'rotation_deg': float(degree), 'shift_x_um': dx, 'shift_y_um': dy,
        'displacement_mean_um': float(displacement.mean()), 'displacement_max_um': float(displacement.max())
    }

In [ ]:
cells = {section: load_cells(section) for section in SECTIONS}
genes = np.flatnonzero(cells['section_04'][1].mean(0) >= 0.3)

# Check on planted transformations (two halves of the cells of section_04)
coords, X = cells['section_04'][0], cells['section_04'][1][:, genes]
half = np.random.default_rng(0).random(len(coords)) < 0.5
for scale, degree, shift in [(1., 0., (60., -40.)), (1.05, 2., (30., 50.)), (0.93, -3., (-80., 20.))]:
    theta, center = np.radians(degree), coords[half].mean(0)
    R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    planted_coords = center + ((coords[~half] - np.asarray(shift)) - center) @ R / scale
    res = get_residual(coords[half], X[half], planted_coords, X[~half])
    print(f"Planted scale {scale:.2f} rotation {degree:+.1f} shift {shift} -> found {res['scale']:.3f}, "
          f"{res['rotation_deg']:+.1f}, ({res['shift_x_um']:.0f}, {res['shift_y_um']:.0f})")

# Consecutive sections
records = []
for fixed_section, moving_section in zip(SECTIONS[:-1], SECTIONS[1:]):
    records.append({
        'pair': f'{fixed_section[-2:]}-{moving_section[-2:]}',
        'gap_um': SPACING * (int(moving_section[-2:]) - int(fixed_section[-2:])),
        **get_residual(
            cells[fixed_section][0], cells[fixed_section][1][:, genes],
            cells[moving_section][0], cells[moving_section][1][:, genes]
        )
    })

slice_df = pd.DataFrame(records)
slice_df.to_csv(os.path.join(outdir, f'LYNX_liver_{SAMPLE_ID}_slice_registration.csv'), index=False)
display(slice_df.round(3))
print(f"Mean displacement over pairs: {slice_df['displacement_mean_um'].mean():.1f} ± {se(slice_df['displacement_mean_um']):.1f} um")